<a href="https://colab.research.google.com/github/FaridRash/brain-ct-hemorrhage-segmentation/blob/main/04_air_filtering_01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Github

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


#Unzipping

In [ ]:
import os

output_dir = '/content/brain_ct_project'
os.makedirs(output_dir, exist_ok=True)

zip_file_path = '/content/drive/MyDrive/brain_ct_project/processed_data_full.zip'

# Unzip the file
!unzip -q "{zip_file_path}" -d "{output_dir}"

print(f"File unzipped to: {output_dir}")
!ls -F "{output_dir}"

File unzipped to: /content/brain_ct_project
test/  train/  val/


In [ ]:
import shutil

destination_dir = '/content/brain_ct_project/'

#label_train.csv
source_file = '/content/drive/MyDrive/brain_ct_project/label_train.csv'
shutil.copy(source_file, destination_dir)

#label_val.csv
source_file = '/content/drive/MyDrive/brain_ct_project/label_val.csv'
shutil.copy(source_file, destination_dir)

#label_test.csv
source_file = '/content/drive/MyDrive/brain_ct_project/label_test.csv'
shutil.copy(source_file, destination_dir)


print(f"File '{source_file}' copied to '{destination_dir}'")
!ls -F "{destination_dir}"

File '/content/drive/MyDrive/brain_ct_project/label_test.csv' copied to '/content/brain_ct_project/'
label_test.csv	label_train.csv  label_val.csv	test/  train/  val/


#Air filtering

In [ ]:
import numpy as np

def is_air_slice(img, air_hu_threshold=-800, air_ratio_threshold=0.9):
    air_pixels = np.sum(img < air_hu_threshold)
    total_pixels = img.size
    air_ratio = air_pixels / total_pixels
    return air_ratio > air_ratio_threshold

In [ ]:
import os
import pandas as pd
import numpy as np

base_path = "/content/brain_ct_project"
splits = ["train", "val", "test"]

for split in splits:

    print(f"\nProcessing {split}...")

    image_dir = os.path.join(base_path, split, "images")
    label_path = os.path.join(base_path, f"label_{split}.csv")

    df = pd.read_csv(label_path)

    keep_rows = []

    for _, row in df.iterrows():

        img_path = os.path.join(image_dir, row["filename"])
        img = np.load(img_path)

        label = row["label"]

        if label == 1:
            keep_rows.append(row)  # ALWAYS keep positives
        else:
            if not is_air_slice(img):
                keep_rows.append(row)

    new_df = pd.DataFrame(keep_rows)

    new_df.to_csv(
        os.path.join(base_path, f"label_{split}_filtered.csv"),
        index=False
    )

    print(f"{split}: {len(df)} → {len(new_df)} kept")


Processing train...
train: 2281 → 2116 kept

Processing val...
val: 241 → 224 kept

Processing test...
test: 292 → 274 kept


In [ ]:
print("Before filtering:")
print(df["label"].value_counts())

print("After filtering:")
print(new_df["label"].value_counts())

Before filtering:
label
0    255
1     37
Name: count, dtype: int64
After filtering:
label
0    237
1     37
Name: count, dtype: int64
